In [1]:
from particles import (get_distance, vels_after_collision, 
                       one_step_simple, init_in_center_grid)
import numpy as np

Функция для инициализация стартовых значений координат и скоростей

In [ ]:
def init_rs_vs(bounds, n_particles, velosity_factor=0.01):

    left_1 = bounds[1] // 6
    right_1 = 2 * left_1

    left_2 = bounds[1] - left_1
    right_2 = bounds[1] - 2 * left_1

    
    coord_values_1 = np.linspace(left_1, right_1, int(np.sqrt(n_particles // 2)))
    coord_values_2 = np.linspace(left_2, right_2, int(np.sqrt(n_particles // 2)))

    rs = np.zeros((n_particles, 2), dtype=np.float16)


    counter = 0
    for i in range(int(np.sqrt(n_particles//2))):
        for j in range(int(np.sqrt(n_particles//2))):
            rs[counter] = coord_values_1[i], coord_values_1[j]
            counter += 1

    for i in range(int(np.sqrt(n_particles//2))):
        for j in range(int(np.sqrt(n_particles//2))):
            rs[counter] = coord_values_2[i], coord_values_2[j]
            counter += 1

    # vs = np.random.uniform(-bounds[1], bounds[1], size=(n_particles, 2)) * velosity_factor
    vs = np.ones((n_particles, 2), dtype=np.float64) * velosity_factor
    vs[n_particles//2:, :] = -1.0 * velosity_factor
    # print(vs)
    # vs[:, 1] = 1
    return rs, vs

In [95]:
3**4

81

In [96]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle, Circle

bounds = 0., 100.   # границы (совпадают как для X так и для Y)
wall_width = 2
N = 2 * 7**2              # число частиц
d = 2.

rs, vs = init_rs_vs(bounds, N, velosity_factor=0.02)

GRAY = "#333"
DARKGRAY = "#222"
LIGHTGRAY = "#777"
figwidth = 10

# Палитра/нормализация для стабильной окраски по скорости
cmap = plt.get_cmap('hot')
speed_norm = mcolors.Normalize(vmin=0, vmax=np.linalg.norm(vs, axis=1).max() + 1e-12)

# Запуск симуляции (PyQt6)

In [100]:
%matplotlib qt

fig, ax = plt.subplots()
fig.set_figwidth(figwidth)
fig.set_figheight(figwidth)
fig.set_facecolor(GRAY)

rs, vs = init_rs_vs(bounds, N, velosity_factor=0.5)


for n in range(500):
    ax.clear()

    # Частицы как круги радиусом d/2
    # colors = np.linalg.norm(vs, axis=1) # Скорости для окраски
    for x, y in rs:
        ax.add_patch(
            Circle(
                (x, y),
                radius=d/2,
                facecolor=LIGHTGRAY,
                edgecolor='none',
                zorder=1
            )
        )

    one_step(rs, vs, d, bounds)

    ax.set_xlim(bounds)
    ax.set_ylim(bounds)
    ax.set_aspect('equal', adjustable='box')
    ax.set_facecolor(DARKGRAY)

    # Убираем подписи и риски на осях
    ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.pause(0.05)

KeyboardInterrupt: 

# Запись анимации в MP4

In [48]:
from matplotlib.animation import FFMpegWriter
import numpy as np
import matplotlib as mpl
import imageio_ffmpeg

plt.ioff()  # выключаем интерактив, чтобы ничего не мигало во время записи
fig, ax = plt.subplots()
fig.set_figwidth(figwidth)
fig.set_figheight(figwidth)
fig.set_facecolor(DARKGRAY)

rs, vs = init_rs_vs(bounds, N, velosity_factor=0.002)


# Настройка видеозаписи
fps = 60
frames = 1200
writer = FFMpegWriter(
    fps=fps,
    codec='libx264',          # H.264
    bitrate=1000,             # можно поднять до 6000–8000 для лучшего качества
    metadata={'title': '2D elastic collisions', 'artist': 'ordevoir'}
)

out_path = "./heat.mp4"
dpi = 150
mpl.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

with writer.saving(fig, out_path, dpi):
    for n in range(frames):
        ax.clear()

        # частицы как круги радиусом d/2
        colors = np.linalg.norm(vs, axis=1)
        for (x, y), c in zip(rs, colors):
            ax.add_patch(
                Circle(
                    (x, y),
                    radius=d/2,
                    facecolor=LIGHTGRAY,
                    edgecolor='none',
                    zorder=1
                )
            )

        one_step(rs, vs, d, bounds)

        ax.set_xlim(bounds)
        ax.set_ylim(bounds)
        ax.set_aspect('equal', adjustable='box')
        ax.set_facecolor(GRAY)
        ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
        for spine in ax.spines.values():
            spine.set_visible(False)

        writer.grab_frame()     # записываем кадр

plt.close(fig)
print(f"Path: {out_path}")

Path: ./heat.mp4


# Триминг ролика

In [2]:
import subprocess
from pathlib import Path
from typing import Union
import imageio_ffmpeg

Time = Union[float, int, str]  # секунды (число) или "HH:MM:SS[.ms]"

def _to_ffmpeg_ts(t: Time) -> str:
    """Привести время к строке HH:MM:SS.mmm для ffmpeg."""
    if isinstance(t, (int, float)):
        ms = int(round((t - int(t)) * 1000))
        s  = int(t) % 60
        m  = (int(t) // 60) % 60
        h  = int(t) // 3600
        return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"
    return t  # уже строка

def trim_mp4(
    in_path: str,
    out_path: str,
    start: Time,
    end: Time,
    accurate: bool = False,
    reencode_video: str = "libx264",  # используется если accurate=True
    reencode_audio: str = "aac",      # используется если accurate=True
    crf: int = 18,                    # качество для H.264 (меньше = лучше)
    preset: str = "medium"            # скорость/сжатие для H.264
) -> None:
    """
    Обрезать видео [start, end). Если accurate=False — быстрое копирование потоков,
    но резка по ближайшим ключевым кадрам. Если accurate=True — точная резка с перекодированием.
    """
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
    in_path = str(Path(in_path))
    out_path = str(Path(out_path))
    ss = _to_ffmpeg_ts(start)
    to = _to_ffmpeg_ts(end)

    # Для ffmpeg удобнее задавать длительность (-t), чем -to, когда используем -ss.
    # Посчитаем длительность в секундах для надёжности.
    def _to_seconds(t: Time) -> float:
        if isinstance(t, (int, float)):
            return float(t)
        h, m, s = t.split(":")
        s = float(s)
        return int(h)*3600 + int(m)*60 + s

    duration = _to_seconds(end) - _to_seconds(start)
    if duration <= 0:
        raise ValueError("end должно быть больше start")

    if accurate:
        # Точный вариант: -ss после -i и перекодирование
        cmd = [
            ffmpeg, "-y",
            "-i", in_path,
            "-ss", ss,
            "-t", f"{duration:.3f}",
            "-map", "0",
            "-c:v", reencode_video, "-crf", str(crf), "-preset", preset,
            "-c:a", reencode_audio, "-b:a", "192k",
            "-movflags", "+faststart",
            out_path
        ]
    else:
        # Быстрый вариант: -ss до -i и копирование потоков (режет по keyframe)
        cmd = [
            ffmpeg, "-y",
            "-ss", ss,
            "-i", in_path,
            "-t", f"{duration:.3f}",
            "-map", "0",
            "-c", "copy",
            "-avoid_negative_ts", "make_zero",
            "-movflags", "+faststart",
            out_path
        ]

    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, capture_output=True, text=True)
    if completed.returncode != 0:
        # Вывести stderr для диагностики
        raise RuntimeError(f"ffmpeg error:\n{completed.stderr}")
    else:
        print(f"Saved: {out_path}")

# Примеры использования:

# 1) Быстро (без перекодирования), с 10.0 c до 25.0 c:
# trim_mp4("./simulation.mp4", "./simulation_trim_fast.mp4", start=10.0, end=25.0, accurate=False)

# 2) Точно (с перекодированием), с 00:00:10.000 до 00:00:25.000:
# trim_mp4("./simulation.mp4", "./simulation_trim_exact.mp4",
#          start="00:00:10.000", end="00:00:25.000", accurate=True, crf=20, preset="faster")


In [15]:
trim_mp4("gas_simulation_compressed.mp4", "gas_simulation_trimmed_2.mp4", 0, 60)

Running: c:\Users\ordevoir\miniconda3\envs\marl\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe -y -ss 00:00:00.000 -i gas_simulation_compressed.mp4 -t 60.000 -map 0 -c copy -avoid_negative_ts make_zero -movflags +faststart gas_simulation_trimmed_2.mp4
Saved: gas_simulation_trimmed_2.mp4


# Симуляция с гистограммой

In [101]:
%matplotlib qt

# Инициализация
rs, vs = init_rs_vs(bounds, N, velosity_factor=0.5)

# Включение интерактивного режима и геометрия
plt.ion()
figwidth = 5
fig_width = figwidth
fig_height = fig_width * 1.5  # верх: ширина; низ: половина ширины
fig = plt.figure(figsize=(fig_width, fig_height))
gs = fig.add_gridspec(2, 1, height_ratios=[2, 1])

ax_sim = fig.add_subplot(gs[0])   # Верхняя панель - симуляция
ax_hist = fig.add_subplot(gs[1])  # Нижняя панель - гистограмма
fig.set_facecolor(GRAY)

# --- Параметры гистограммы: фиксированные бины и фиксированный верх по старту ---
n_bins = 15
bins = np.linspace(0, 2, n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bin_widths  = np.diff(bins)

speeds_init = np.linalg.norm(vs, axis=1)
init_counts, _ = np.histogram(speeds_init, bins=bins)
y_max = max(1, init_counts.max())  # фиксируем верхнюю границу по начальному максимуму

for n in range(500):
    ax_sim.clear()
    ax_hist.clear()
    
    # === ВЕРХНЯЯ ПАНЕЛЬ: частицы ===
    for x, y in rs:
        ax_sim.add_patch(Circle((x, y), radius=d/2, facecolor=LIGHTGRAY, edgecolor='none', zorder=1))
    ax_sim.set_xlim(bounds)
    ax_sim.set_ylim(bounds)
    ax_sim.set_aspect('equal', adjustable='box')
    ax_sim.set_facecolor(DARKGRAY)
    ax_sim.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    for spine in ax_sim.spines.values():
        spine.set_visible(False)

    # === НИЖНЯЯ ПАНЕЛЬ: гистограмма абсолютных счётчиков ===
    speeds = np.linalg.norm(vs, axis=1)
    counts, _ = np.histogram(speeds, bins=bins)   # абсолютные значения

    # 1 пиксельный зазор между столбиками (переводим пиксели в дата-юниты)
    ax_hist.set_xlim(0, 2)
    fig_w_px = fig.get_figwidth() * fig.dpi
    axis_px_width = ax_hist.get_position().width * fig_w_px
    data_per_px = (ax_hist.get_xlim()[1] - ax_hist.get_xlim()[0]) / axis_px_width
    gap_data = data_per_px * 1.0  # 1 px -> data units
    bar_widths = np.maximum(bin_widths - gap_data, 0.0)

    ax_hist.bar(bin_centers, counts, width=bar_widths, align='center',
                color=LIGHTGRAY, edgecolor='none')
    ax_hist.set_ylim(0, y_max)

    # Чистое поле без подписей/рамок + надпись в верхнем правом углу
    ax_hist.set_facecolor(DARKGRAY)
    ax_hist.grid(False)
    ax_hist.set_title(''); ax_hist.set_xlabel(''); ax_hist.set_ylabel('')
    ax_hist.set_xticks([]); ax_hist.set_yticks([])
    ax_hist.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    for spine in ax_hist.spines.values():
        spine.set_visible(False)

    ax_hist.text(0.98, 0.98, "Velocity\nDistribution",
                 transform=ax_hist.transAxes, ha='right', va='top',
                 color=LIGHTGRAY)

    # Шаг симуляции
    one_step(rs, vs, d, bounds)

    plt.tight_layout()
    fig.canvas.draw()
    fig.canvas.flush_events()
    plt.pause(0.01)

plt.ioff()
plt.show()


KeyboardInterrupt: 

# Сохранение в MP4

In [106]:
from matplotlib.animation import FFMpegWriter
import numpy as np
import matplotlib as mpl
import imageio_ffmpeg

# Выключаем интерактивный режим
plt.ioff()

# Инициализация
rs, vs = init_rs_vs(bounds, N, velosity_factor=0.5)

# Геометрия фигуры
figwidth = 5
fig_width = figwidth
fig_height = fig_width * 1.5
fig = plt.figure(figsize=(fig_width, fig_height))
gs = fig.add_gridspec(2, 1, height_ratios=[2, 1])

ax_sim = fig.add_subplot(gs[0])   # Верхняя панель - симуляция
ax_hist = fig.add_subplot(gs[1])  # Нижняя панель - гистограмма
fig.set_facecolor(GRAY)

# --- Параметры гистограммы: фиксированные бины и фиксированный верх по старту ---
n_bins = 15
bins = np.linspace(0, 2, n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bin_widths = np.diff(bins)

speeds_init = np.linalg.norm(vs, axis=1)
init_counts, _ = np.histogram(speeds_init, bins=bins)
y_max = max(1, init_counts.max())  # фиксируем верхнюю границу

# Настройка видеозаписи
fps = 60
frames = 100
writer = FFMpegWriter(
    fps=fps,
    codec='libx264',
    bitrate=2000,
    metadata={'title': '2D Gas Simulation', 'artist': 'your_name'}
)

out_path = "./gas_simulation.mp4"
dpi = 100
mpl.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

with writer.saving(fig, out_path, dpi):
    for n in range(frames):
        ax_sim.clear()
        ax_hist.clear()
        
        # === ВЕРХНЯЯ ПАНЕЛЬ: частицы ===
        for x, y in rs:
            ax_sim.add_patch(Circle((x, y), radius=d/2, facecolor=LIGHTGRAY, 
                                   edgecolor='none', zorder=1))
        ax_sim.set_xlim(bounds)
        ax_sim.set_ylim(bounds)
        ax_sim.set_aspect('equal', adjustable='box')
        ax_sim.set_facecolor(DARKGRAY)
        ax_sim.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
        for spine in ax_sim.spines.values():
            spine.set_visible(False)

        # === НИЖНЯЯ ПАНЕЛЬ: гистограмма абсолютных счётчиков ===
        speeds = np.linalg.norm(vs, axis=1)
        counts, _ = np.histogram(speeds, bins=bins)

        # 1 пиксельный зазор между столбиками
        ax_hist.set_xlim(0, 2)
        fig_w_px = fig.get_figwidth() * dpi  # используем dpi из writer
        axis_px_width = ax_hist.get_position().width * fig_w_px
        data_per_px = (ax_hist.get_xlim()[1] - ax_hist.get_xlim()[0]) / axis_px_width
        gap_data = data_per_px * 1.0
        bar_widths = np.maximum(bin_widths - gap_data, 0.0)

        ax_hist.bar(bin_centers, counts, width=bar_widths, align='center',
                    color=LIGHTGRAY, edgecolor='none')
        ax_hist.set_ylim(0, y_max)

        # Чистое поле без подписей/рамок
        ax_hist.set_facecolor(DARKGRAY)
        ax_hist.grid(False)
        ax_hist.set_title(''); ax_hist.set_xlabel(''); ax_hist.set_ylabel('')
        ax_hist.set_xticks([]); ax_hist.set_yticks([])
        ax_hist.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
        for spine in ax_hist.spines.values():
            spine.set_visible(False)

        ax_hist.text(0.98, 0.98, "Velocity\nDistribution",
                     transform=ax_hist.transAxes, ha='right', va='top',
                     color=LIGHTGRAY)

        # Шаг симуляции
        one_step(rs, vs, d, bounds)

        plt.tight_layout()
        writer.grab_frame()  # записываем кадр

plt.close(fig)
print(f"Path: {out_path}")

Path: ./gas_simulation.mp4


In [14]:
import subprocess
import os
import imageio_ffmpeg

def compress_video_ffmpeg(input_path, output_path, crf=18):
    """
    Сжатие видео с использованием FFmpeg и кодека H.264
    
    Параметры:
    - input_path: путь к входному файлу
    - output_path: путь к выходному файлу
    - crf: Constant Rate Factor (постоянный фактор скорости), 0-51
          0 = без потерь, 23 = по умолчанию, 51 = худшее качество
          18 = визуально без потерь (visually lossless)
    """
    # Получаем путь к FFmpeg из imageio_ffmpeg
    ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
    
    command = [
        ffmpeg_path,                # используем путь к FFmpeg
        '-i', input_path,
        '-c:v', 'libx264',
        '-crf', str(crf),
        '-preset', 'slow',
        '-c:a', 'aac',
        '-b:a', '128k',
        '-movflags', '+faststart',
        '-y',
        output_path
    ]
    
    try:
        result = subprocess.run(command, check=True, capture_output=True, text=True)
        
        # Статистика сжатия
        original_size = os.path.getsize(input_path) / (1024 * 1024)  # МБ
        compressed_size = os.path.getsize(output_path) / (1024 * 1024)
        compression_ratio = (1 - compressed_size / original_size) * 100
        
        print(f"Оригинальный размер: {original_size:.2f} МБ")
        print(f"Сжатый размер: {compressed_size:.2f} МБ")
        print(f"Сжатие: {compression_ratio:.1f}%")
        
    except subprocess.CalledProcessError as e:
        print(f"Ошибка при сжатии: {e.stderr}")
    except FileNotFoundError:
        print("FFmpeg не найден. Установите: pip install imageio-ffmpeg")

# Использование
compress_video_ffmpeg("./gas_simulation3.mp4", "./gas_simulation_compressed.mp4", crf=18)

Оригинальный размер: 39.59 МБ
Сжатый размер: 14.42 МБ
Сжатие: 63.6%


FileNotFoundError: [WinError 2] The system cannot find the file specified